In [ ]:
import os
import sys
import requests
import pandas as pd
import time
from collections import deque
from dotenv import load_dotenv

# --- 1. Environment & Directory Setup ---
notebook_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in locals() else os.getcwd()
config_dir = os.path.abspath(os.path.join(notebook_dir, "..", "..", "config"))
raw_data_dir = os.path.abspath(os.path.join(notebook_dir, "..", "..", "data", "raw"))
interim_data_dir = os.path.abspath(os.path.join(notebook_dir, "..", "..", "data", "interim"))
error_log_dir = os.path.abspath(os.path.join(interim_data_dir, "logs", "errors"))

os.makedirs(raw_data_dir, exist_ok=True)
os.makedirs(error_log_dir, exist_ok=True)

load_dotenv(os.path.join(config_dir, ".env")) 
contact_email = os.getenv("CONTACT_EMAIL", "anonymous@example.com")
sys.path.append(config_dir)

from ingest_schema_manager import get_registry_info, finalize_row, COLUMN_ORDER

SOURCE_PREFIX = "WIKIDATA"
registry_data = get_registry_info(prefix=SOURCE_PREFIX, config_dir=config_dir)
SOURCE_NAME = registry_data['Source_Name']
BASE_URI = registry_data['Base_URI']

raw_ingest_file = os.path.join(raw_data_dir, f"raw_{SOURCE_PREFIX.lower()}.csv")
hard_failure_file = os.path.join(error_log_dir, f"{SOURCE_PREFIX.lower()}_hard_failures.csv")

HEADERS = {
    'User-Agent': f'ReligiousMappingProject/1.0 (mailto:{contact_email})',
    'Accept': 'application/json'
}

# --- 2. Scope & Guardrails ---
TARGET_SEEDS = ['Q5043']  # Christianity
MAX_DEPTH = 3
CORE_ROOTS = {'Q5043', 'Q9174'}  # Christianity, Religion

# Map Wikidata Properties to source_registry prefixes
CROSSWALK_MAP = {
    'P486': 'MeSH', 
    'P244': 'LCSH', 
    'P1014': 'AAT', 
    'P5806': 'SNOMED', 
    'P343': 'LOINC', 
    'P1036': 'DDC', 
    'P1190': 'UDC', 
    'P3348': 'LCDGT', 
    'P10165': 'TGM',
    'P12107': 'ELSST',    # New: European Language Social Science Thesaurus
    'P3916': 'UNESCO',    # New: UNESCO Thesaurus
    'P2163': 'FAST',      # New: Faceted Application of Subject Terminology
    'P227': 'GND',        # New: Integrated Authority File (Germany)
    'P3921': 'EUROVOC',   # New: EU Multilingual Thesaurus
    'P10283': 'OpenAlex'  # OpenAlex
}

# --- 3. Global State ---
if os.path.exists(raw_ingest_file):
    existing_df = pd.read_csv(raw_ingest_file, dtype={'Concept_ID': str})
    if list(existing_df.columns) != COLUMN_ORDER:
        print(f"[!] Schema mismatch. Deleting {os.path.basename(raw_ingest_file)}...")
        os.remove(raw_ingest_file)
        global_do_not_crawl = set()
    else:
        global_do_not_crawl = set(existing_df['Concept_ID'].dropna().tolist())
else:
    global_do_not_crawl = set()

path_cache = {}
api_cache = {}
ancestry_cache = {}
retry_queue = []

# --- 4. API Wrappers ---
def query_sparql(query_string):
    """Executes a Wikidata SPARQL query."""
    url = "https://query.wikidata.org/sparql"
    for attempt in range(1, 4):
        try:
            res = requests.get(url, headers=HEADERS, params={'query': query_string, 'format': 'json'}, timeout=30)
            if res.status_code == 200:
                return res.json()
            elif res.status_code == 429:
                print(f"  [!] Rate limited by SPARQL. Backing off...")
        except requests.exceptions.RequestException:
            pass
        time.sleep(2 ** attempt)
    return None

def fetch_entity_metadata(qid):
    """Uses the Action API to get specific node metadata efficiently."""
    if qid in api_cache:
        return api_cache[qid]
    
    url = "https://www.wikidata.org/w/api.php"
    params = {
        'action': 'wbgetentities', 'ids': qid, 'format': 'json',
        'props': 'labels|descriptions|aliases|claims'
    }
    
    for attempt in range(1, 4):
        try:
            res = requests.get(url, headers=HEADERS, params=params, timeout=30)
            if res.status_code == 200:
                data = res.json().get('entities', {}).get(qid)
                if data:
                    api_cache[qid] = data
                    return data
        except requests.exceptions.RequestException:
            pass
        time.sleep(2 ** attempt)
    return None

def get_children(qid):
    """Finds immediate P279 subclasses via SPARQL."""
    query = f"""
    SELECT ?child WHERE {{
      ?child wdt:P279 wd:{qid}.
    }}
    """
    data = query_sparql(query)
    if not data:
        return []
    
    children = []
    for item in data.get('results', {}).get('bindings', []):
        child_url = item.get('child', {}).get('value', '')
        child_qid = child_url.split('/')[-1]
        if child_qid.startswith('Q'):
            children.append(child_qid)
    return children

# --- 5. BFS & Absolute Ancestry Resolution ---
def get_best_label(metadata, default_qid):
    """Helper to safely extract an English label, or the first available fallback."""
    labels = metadata.get('labels', {})
    if 'en' in labels:
        return labels['en'].get('value', default_qid)
    elif labels:
        fallback_lang = list(labels.keys())[0]
        return labels[fallback_lang].get('value', default_qid)
    return default_qid

def get_absolute_ancestry(qid, depth=0):
    """Recursive upward climb to the absolute top of the Wikidata tree."""
    if qid in ancestry_cache: 
        return ancestry_cache[qid]
    
    # Prevent infinite loops in Wikidata's highly abstract upper ontology
    if depth > 10:  
        return ""
        
    metadata = fetch_entity_metadata(qid)
    if not metadata: 
        return ""
    
    label = get_best_label(metadata, qid)  # Use curr_qid in get_best_path
    parents = metadata.get('claims', {}).get('P279', [])
    
    if parents:
        # Greedily take the first P279 parent to represent the upper taxonomy
        p_qid = parents[0].get('mainsnak', {}).get('datavalue', {}).get('value', {}).get('id')
        if p_qid:
            parent_path = get_absolute_ancestry(p_qid, depth + 1)
            full_path = f"{parent_path} > {label}" if parent_path else label
            ancestry_cache[qid] = full_path
            return full_path
            
    ancestry_cache[qid] = label
    return label

def get_best_path(start_qid):
    """BFS for the shortest path to a core root, then climbs to absolute root."""
    if start_qid in path_cache:
        return path_cache[start_qid]
        
    queue = deque([(start_qid, [], 0)])
    visited = {start_qid}
    
    while queue:
        curr_qid, path_labels, depth = queue.popleft()
        if depth > 10: 
            continue
            
        metadata = fetch_entity_metadata(curr_qid)
        if not metadata:
            continue
            
        label = get_best_label(metadata, curr_qid)  # Use curr_qid in get_best_path
        
        # When we hit a recognized domain root, trigger the absolute climb
        if curr_qid in CORE_ROOTS:
            absolute_top = get_absolute_ancestry(curr_qid)
            # Stitch the absolute top to the specific path we found
            final_path = f"{absolute_top} > {' > '.join(path_labels[::-1])}" if path_labels else absolute_top
            path_cache[start_qid] = final_path
            return final_path
            
        # Extract parents (P279)
        claims = metadata.get('claims', {})
        parents = claims.get('P279', [])
        
        for p in parents:
            p_qid = p.get('mainsnak', {}).get('datavalue', {}).get('value', {}).get('id')
            if p_qid and p_qid not in visited:
                visited.add(p_qid)
                new_labels = list(path_labels)
                new_labels.append(label)
                queue.append((p_qid, new_labels, depth + 1))
                
    return None

# --- 6. Main Extraction Logic ---
def process_node(qid, p_id="", current_depth=0, is_retry_pass=False):
    is_already_ingested = qid in global_do_not_crawl
    
    # Translucent Crawl: Stop if ingested AND at max depth
    if is_already_ingested and current_depth >= MAX_DEPTH:
        return

    metadata = fetch_entity_metadata(qid)
    if not metadata:
        if not is_retry_pass:
            retry_queue.append({'uri': qid, 'p_id': p_id, 'depth': current_depth})
        return

    if not is_already_ingested:
        # 1. Multi-language Label & Description Resolution
        primary_label = get_best_label(metadata, qid)
        
        # Hard Drop Condition: No labels exist in any language
        if not primary_label:
            print(f"\n  [!] Dropping {qid}: No labels found in any language.")
            if not is_retry_pass:
                global_do_not_crawl.add(qid)
            return
            
        # Description matches the label's language
        description = ""
        labels = metadata.get('labels', {})
        if 'en' in labels:
            description = metadata.get('descriptions', {}).get('en', {}).get('value', '')
        elif labels:
            fallback_lang = list(labels.keys())[0]
            description = metadata.get('descriptions', {}).get(fallback_lang, {}).get('value', '')

        # Synonyms (still defaulting to English aliases if they exist, else blank)
        aliases = metadata.get('aliases', {}).get('en', [])
        synonyms = " | ".join([a.get('value') for a in aliases]) if aliases else ""
        
        # 2. Translation Check
        has_translation = "yes" if len(metadata.get('labels', {}).keys()) > 1 else ""
        
        # 3. Ancestry & Parent IDs
        valid_path = get_best_path(qid) or primary_label
        
        claims = metadata.get('claims', {})
        parents = [p.get('mainsnak', {}).get('datavalue', {}).get('value', {}).get('id') 
                   for p in claims.get('P279', [])]
        parent_ids_str = " | ".join(filter(None, parents)) if parents else str(p_id)
        
        # 4. Crosswalks
        crosswalks_list = []
        for prop, prefix in CROSSWALK_MAP.items():
            ext_ids = claims.get(prop, [])
            for ext in ext_ids:
                val = ext.get('mainsnak', {}).get('datavalue', {}).get('value')
                if val:
                    crosswalks_list.append(f"{prefix}:{val}")
        crosswalk_str = " | ".join(crosswalks_list)

        print(f"\r[DEPTH {current_depth}] Ingesting: {primary_label[:50]:<50}", end="", flush=True)

        extracted = {
            'Source_System': SOURCE_NAME,
            'Primary_Label': primary_label,
            'CURIE': f"WIKIDATA:{qid}",
            'Formal_Label': "", 
            'Concept_Type': 'wikibase:Item',
            'Hierarchy_Path': valid_path,
            'Synonyms': synonyms,
            'Description': description,
            'Parent_IDs': parent_ids_str,
            'Concept_ID': qid,
            'URI': f"http://www.wikidata.org/entity/{qid}",
            'Has_Translation': has_translation,
            'Status': 'active',
            'Crosswalks': crosswalk_str
        }
        
        pd.DataFrame([finalize_row(extracted)])[COLUMN_ORDER].to_csv(
            raw_ingest_file, mode='a', index=False, header=not os.path.isfile(raw_ingest_file), encoding='utf-8-sig'
        )
        global_do_not_crawl.add(qid)
    else:
        label = metadata.get('labels', {}).get('en', {}).get('value', qid)
        print(f"\r[DEPTH {current_depth}] Passing Through: {label[:45]:<45}", end="", flush=True)

    # Downward Recursion
    if current_depth < MAX_DEPTH:
        children = get_children(qid)
        for child_qid in children:
            process_node(child_qid, p_id=qid, current_depth=current_depth + 1, is_retry_pass=is_retry_pass)

# --- 7. Execution ---
print(f"Starting Wikidata API Crawl (Target Depth: {MAX_DEPTH})...\n")
for seed in TARGET_SEEDS:
    process_node(seed, current_depth=0)

if retry_queue:
    print(f"\n\nPhase 2 Cleanup. Retrying {len(retry_queue)} failed nodes...")
    current_retries = list(retry_queue)
    retry_queue.clear()
    for task in current_retries:
        process_node(task['uri'], p_id=task['p_id'], current_depth=task['depth'], is_retry_pass=True)

print(f"\n\nDone. Final data appended to {raw_ingest_file}")